**Daily Challenge : AI Agent for Emergency Medical Dispatch**


**1. Document de conception**

**Définition de l'environnement**
* Entrées (Perceptions) :

> Transcription vocale ou textuelle des symptômes de l'appelant et de la description de la situation.

> Localisation de l'appelant (coordonnées GPS du téléphone ou adresse saisie manuellement).

> Identité de l'appelant et tout antécédent médical disponible (par exemple, à partir d'un système de dossier médical lié, si le consentement est donné).


**Liste des outils et interfaces**

* API de vérification des symptômes (par exemple, Infermedica)
> Données consommées : symptoms_list (analysé à partir de la transcription de l'appel)
> Données retournées : possible_conditions (une liste de conditions médicales potentielles), preliminary_triage_score

* API du système de planification des ambulances
> Données consommées : surgency_level (Élevé, Moyen, Faible), patient_location, required_resources (par exemple, ambulancier, technicien médical d'urgence)
> Données retournées : pdispatch_status (par exemple, "en route"), ETA (heure d'arrivée estimée), ambulance_ID

* Modèle de triage médical (LLM affiné)
> Données consommées : full_call_transcript, symptoms_list, preliminary_triage_score de l'API de vérification des symptômes
> Données retournées : final_urgency_score (échelle de 1 à 10), recommended_action

* Services SIG/de localisation
> Données consommées : address ou GPS_coordinates
> Données retournées : parsed_address, nearest_hospital_location, nearest_urgent_care_location

**Schéma de gestion de l'état**

Un schéma JSON pour gérer l'état d'un seul appel :

In [ ]:
{
  "call_id": "unique_call_identifier",
  "caller_info": {
    "name": "string",
    "phone_number": "string",
    "location": {
      "address": "string",
      "gps_coordinates": "string"
    },
    "medical_history_available": "boolean"
  },
  "symptoms_reported": [
    {
      "symptom": "string",
      "reported_severity": "string"
    }
  ],
  "triage_results": {
    "urgency_score": "number (1-10)",
    "urgency_level": "string (High, Medium, Low)",
    "recommended_action": "string"
  },
  "actions_taken": [
    {
      "timestamp": "string (ISO 8601)",
      "action_type": "string (e.g., dispatch_ambulance, advise_self_care)",
      "action_details": "json_object"
    }
  ],
  "current_status": "string (e.g., 'analyzing', 'awaiting_dispatch', 'call_complete')"
}

**Organigramme de prise de décision (pseudocode)**

FUNCTION handle_911_call(call_transcript, caller_location, medical_history):
  // 1. Initialize State
  call_state = create_new_state(call_transcript, caller_location, medical_history)

  // 2. Perception & Information Gathering
  symptoms = parse_symptoms(call_transcript)
  call_state.symptoms_reported = symptoms

  // 3. Reasoning & Urgency Determination
  initial_triage_data = Symptom_Checker_API.query(symptoms)
  final_triage_data = Medical_Triage_Model.query(call_state.triage_results)
  call_state.triage_results = final_triage_data

  // 4. Action Based on Urgency
  urgency_level = call_state.triage_results.urgency_level

  IF urgency_level IS "High":
    // Immediate dispatch
    dispatch_details = Ambulance_Scheduling_System_API.dispatch(
      urgency_level,
      call_state.caller_info.location,
      required_resources="paramedic_unit"
    )
    call_state.actions_taken.add(action_type="dispatch_ambulance", details=dispatch_details)
    agent_speak("Une ambulance est en route. Veuillez rester en ligne...")

  ELSE IF urgency_level IS "Medium":
    // Advise nearest urgent care
    urgent_care_location = GIS_Location_Services.find_nearest_urgent_care(call_state.caller_info.location)
    call_state.actions_taken.add(action_type="advise_urgent_care", details=urgent_care_location)
    agent_speak("Votre état semble stable. Le centre de soins urgents le plus proche est à {location}...")

  ELSE IF urgency_level IS "Low":
    // Provide self-care instructions
    instructions = get_self_care_instructions(symptoms)
    call_state.actions_taken.add(action_type="provide_self_care_instructions", details=instructions)
    agent_speak("Cette condition peut probablement être gérée à la maison. Suivez ces étapes...")

  // 5. Update and Finalize State
  call_state.current_status = "call_complete"
  save_state(call_state)

  RETURN call_state.actions_taken

**2. Classification et comparaison d'agents**


**Architecture choisie : Agent hybride**

J'ai choisi une architecture d'Agent hybride.

Justification : un agent hybride est le choix le plus approprié pour ce scénario à haut risque. Il utilise une approche délibérative pour le processus de triage de base, où il planifie et raisonne à travers un flux en plusieurs étapes (analyse des symptômes, interrogation des API, obtention d'un score d'urgence). Cela garantit une décision approfondie et fiable. Cependant, il intègre également des éléments réactifs. Par exemple, si un appelant signale soudainement un changement critique et potentiellement mortel (par exemple, "Il ne respire plus !"), l'agent doit réagir en annulant son processus délibératif et en déclenchant immédiatement l'envoi d'une ambulance, plutôt que d'attendre la fin de la séquence de triage complète. Il utilise sa gestion de l'état pour maintenir une mémoire complète de l'appel, ce qui lui permet de se construire une image complète de la situation au fil du temps, ce qui est crucial pour la sécurité et la précision.


**Comparaison avec un deuxième type d'agent : agent réactif**

Si nous devions concevoir un agent réactif pur, la conception changerait considérablement.

* **Gestion de la mémoire :** un agent réactif aurait peu ou pas de mémoire à long terme. Il ne se souviendrait que de la dernière entrée pour décider de sa prochaine action. Le schéma de gestion de l'état ci-dessus serait en grande partie abandonné. Il ne se souviendrait pas des symptômes ou des actions précédentes, mais seulement de l'action en cours.

* **Étapes de planification :** il n'y aurait pas de planification. Un agent réactif fonctionnerait sur un simple ensemble de règles condition-action. Par exemple, si "douleur thoracique" est détecté, alors l'agent envoie une ambulance (dispatch_ambulance). Il n'effectuerait pas le processus en plusieurs étapes d'interrogation d'un vérificateur de symptômes ou d'un modèle de triage.

* **Stratégie d'invocation des outils :** les outils seraient invoqués directement en fonction des mots-clés déclencheurs plutôt que d'une décision raisonnée. Si l'agent perçoit le mot-clé "saignement", il appellerait immédiatement l'outil dispatch_ambulance sans aucun autre contexte ou analyse de la gravité.


**Compromis :**

* **Vitesse : **un agent réactif serait plus rapide car il évite la surcharge de la délibération et des appels d'API pour une analyse complexe. Il enverrait probablement une ambulance plus rapidement dans les urgences évidentes.

* **Fiabilité :** l'agent hybride est plus fiable et plus sûr dans les cas ambigus. Un agent réactif pourrait mal classer une crise de panique comme une crise cardiaque basée uniquement sur des mots-clés, entraînant des envois inutiles et une surcharge des ressources. À l'inverse, il pourrait ne pas détecter un problème sous-jacent grave si les symptômes de l'appelant ne sont pas évidents.

* **Intelligence :** l'agent hybride est bien plus intelligent car il peut intégrer des informations provenant de plusieurs sources, planifier une ligne de conduite et s'adapter en fonction d'une compréhension globale de la situation. Un agent réactif est limité à des réponses simples et préprogrammées et n'a pas la capacité de gérer les nuances ou les informations contradictoires.

3. **Réponses aux questions de réflexion**


**Que se passe-t-il si votre agent ne maintient pas son état ?**

Si l'agent ne maintenait pas son état, il serait dangereusement inefficace. L'agent traiterait chaque nouvelle déclaration de l'appelant comme un événement complètement nouveau, manquant du contexte crucial de la conversation en cours. Par exemple, si l'appelant dit d'abord "J'ai une douleur thoracique" et que l'agent commence un processus de triage, mais que l'appelant dit ensuite "J'ai aussi le souffle court", un agent sans état commencerait un tout nouveau processus de triage isolé pour le deuxième symptôme. Il ne parviendrait pas à combiner ces deux symptômes critiques, qui ensemble indiquent fortement une crise cardiaque, ce qui pourrait conduire à un score d'urgence faible et à un résultat catastrophique. La gestion de l'état est le fondement d'une interaction cohérente et sûre dans un scénario d'urgence.


**Pourquoi les outils externes (API, modèles) sont-ils essentiels dans un scénario de répartition EMR ?**

Les outils externes sont absolument essentiels car un agent d'IA, en particulier celui construit sur un grand modèle de langage, ne peut pas être une source de vérité médicale par lui-même. La fonction principale de l'agent est d'orchestrer et d'interpréter, pas de diagnostiquer. L'API de vérification des symptômes et le Modèle de triage médical fournissent les connaissances spécialisées et validées médicalement requises pour un triage précis. Le système de planification des ambulances est la composante critique de l'action, permettant à l'agent d'agir sur le monde réel en envoyant des ressources physiques. S'appuyer sur les connaissances générales d'un grand modèle de langage pour des conseils médicaux serait irresponsable et très peu fiable. Ces outils externalisent les tâches spécifiques au domaine et à haut risque vers des systèmes dédiés et vérifiés, permettant à l'agent d'IA de se concentrer sur son rôle d'orchestrateur de la perception, du raisonnement et de l'action.

**deliverables.md**


# Agent IA pour la répartition médicale d'urgence : Document de conception et d'analyse

---

## 1. Document de conception

### Définition de l'environnement

L'agent perçoit les entrées suivantes :
* **Transcription vocale ou textuelle** des symptômes et de la situation de l'appelant.
* **Localisation de l'appelant** (GPS ou adresse saisie).
* **Identité et antécédents médicaux** (si disponibles).

### Liste des outils et interfaces

| Nom de l'outil | Données consommées | Données retournées |
| :--- | :--- | :--- |
| **API de vérification des symptômes** | `symptoms_list` | `possible_conditions`, `preliminary_triage_score` |
| **API du système de planification des ambulances** | `urgency_level`, `patient_location`, `required_resources` | `dispatch_status`, `ETA`, `ambulance_ID` |
| **Modèle de triage médical** | `full_call_transcript`, `symptoms_list`, `preliminary_triage_score` | `final_urgency_score`, `recommended_action` |
| **Services SIG/de localisation** | `address` ou `GPS_coordinates` | `parsed_address`, `nearest_hospital_location` |

### Schéma de gestion de l'état (JSON)

```json
{
  "call_id": "unique_call_identifier",
  "caller_info": {
    "name": "string",
    "phone_number": "string",
    "location": {
      "address": "string",
      "gps_coordinates": "string"
    }
  },
  "symptoms_reported": [
    {
      "symptom": "string",
      "reported_severity": "string"
    }
  ],
  "triage_results": {
    "urgency_score": "number (1-10)",
    "urgency_level": "string (High, Medium, Low)",
    "recommended_action": "string"
  },
  "actions_taken": [
    {
      "timestamp": "string (ISO 8601)",
      "action_type": "string",
      "action_details": "json_object"
    }
  ],
  "current_status": "string"
}


**Organigramme de prise de décision (Pseudocode)**


FUNCTION handle_911_call(call_transcript, caller_location, medical_history):
  call_state = create_new_state(...)
  symptoms = parse_symptoms(call_transcript)
  call_state.symptoms_reported = symptoms

  initial_triage_data = Symptom_Checker_API.query(symptoms)
  final_triage_data = Medical_Triage_Model.query(call_state.triage_results)
  call_state.triage_results = final_triage_data

  urgency_level = call_state.triage_results.urgency_level

  IF urgency_level IS "High":
    dispatch_details = Ambulance_Scheduling_System_API.dispatch(...)
    agent_speak("Une ambulance est en route. Veuillez rester en ligne...")

  ELSE IF urgency_level IS "Medium":
    urgent_care_location = GIS_Location_Services.find_nearest_urgent_care(...)
    agent_speak("Votre état semble stable. Le centre de soins urgents le plus proche est à {location}...")

  ELSE IF urgency_level IS "Low":
    instructions = get_self_care_instructions(symptoms)
    agent_speak("Cette condition peut probablement être gérée à la maison. Suivez ces étapes...")

  call_state.current_status = "call_complete"
  save_state(call_state)
  RETURN call_state.actions_taken

2. **Classification et comparaison de l'agent**


**Architecture choisie : Agent hybride**
J'ai choisi une architecture d'Agent hybride.

Cette architecture est la plus adaptée pour un scénario d'urgence car elle combine le meilleur des deux mondes : une approche délibérative pour un triage sûr et fiable, et une approche réactive pour les situations critiques qui nécessitent une réponse immédiate. La gestion de l'état permet une mémoire complète de l'appel, ce qui est essentiel pour une prise de décision éclairée.


**Comparaison avec un Agent réactif**

* Gestion de la mémoire
> Agent Hybride (Choisi) : maintient un état complet de l'appel (symptômes, actions, etc.).
> Agent Réactif : peu ou pas de mémoire, ne se souvient que de la dernière entrée.


* Planification
> Agent Hybride (Choisi) : raisonne en plusieurs étapes (triage, consultation d'outils) avant d'agir.
> Agent Réactif : agit immédiatement en fonction de déclencheurs simples (ex: "douleur thoracique" -> envoi d'ambulance).


*Stratégie d'outil
> Agent Hybride (Choisi) : orchestre plusieurs outils de manière séquentielle pour une analyse approfondie.
> Agent Réactif : appelle des outils directement sur la base de mots-clés, sans analyse de contexte.


* **Compromis**

> **Vitesse :** l'agent réactif est plus rapide, mais potentiellement moins précis.

> **Fiabilité** : l'agent hybride est plus fiable car il effectue une analyse plus approfondie, évitant les erreurs de sur-triage ou de sous-triage.

> **Intelligence** : l'agent hybride est plus intelligent, capable de gérer des situations complexes et des nuances, contrairement à l'agent réactif qui se limite à des règles simples.



3. **Réponses aux questions de réflexion**


**Impact du manque de gestion de l'état**

Sans gestion de l'état, l'agent perdrait le contexte de la conversation. Il traiterait chaque phrase de l'appelant comme un événement isolé, ce qui pourrait conduire à une évaluation erronée. Par exemple, il ne pourrait pas combiner les symptômes qui se manifestent au fil de l'appel (ex : "douleur thoracique" puis "essoufflement"), ce qui est crucial pour identifier une urgence grave.

**Rôle des outils externes dans la répartition à haut risque**

Les outils externes (APIs, modèles spécialisés) sont indispensables car ils apportent la connaissance médicale validée et la capacité d'action concrète que l'agent seul ne possède pas. L'agent ne fait pas de diagnostic médical par lui-même, il agit comme un orchestrateur intelligent qui utilise des outils fiables pour analyser les symptômes, déterminer l'urgence et déclencher l'intervention appropriée. C'est le moyen le plus sûr et le plus fiable de garantir des réponses précises dans des situations de vie ou de mort.